# SAD Detection on the APL Starter Kit Dataset

Spectral Anomaly Detection (PCA reconstruction error) applied to the
APL Starter Kit pre-binned spectral data.

**Workflow:** load background files → train SAD → calibrate threshold →
run on a source file → visualize.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from gammaflow import SpectralTimeSeries
from gammaflow.algorithms import SADDetector
from gammaflow.datasets import APLStarterKitDataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

APL_STARTER_DATA_DIR = '/path/to/Starter Kit Data Set'
DETECTOR = 0

# Match TopCoder example: 512 bins over 0-3000 keV, 5-second integration
ENERGY_BINS = 512
ENERGY_RANGE = (0, 3000)
INTEGRATION_TIME = 5.0

ds = APLStarterKitDataset(APL_STARTER_DATA_DIR)
print(ds)

## Load Background Data & Train SAD

In [ ]:
bg_files = ds.list_files('background')
print(f"{len(bg_files)} background files")

new_energy_edges = np.linspace(ENERGY_RANGE[0], ENERGY_RANGE[1], ENERGY_BINS + 1)

bg_time_series = []
for fname in tqdm(bg_files, desc='Loading background'):
    ts, _ = ds.load_file(fname, split='background', detector=DETECTOR)
    ts = ts.apply_to_each(lambda s: s.rebin_energy(new_energy_edges))
    ts = ts.rebin_time(INTEGRATION_TIME)
    bg_time_series.append(ts)

total_bg_spectra = sum(len(ts) for ts in bg_time_series)
print(f"Total background spectra: {total_bg_spectra}")

In [ ]:
detector = SADDetector(
    n_components=5,
    normalize=False,
    aggregation_gap=2.0,
)

detector.fit(bg_time_series[0])

var_ratios = detector.get_explained_variance_ratio()
cum_var = detector.get_cumulative_variance_explained()
print("Explained variance by component:")
for i, v in enumerate(var_ratios, 1):
    print(f"  PC{i}: {v*100:.2f}%")
print(f"Cumulative: {cum_var*100:.2f}%")

## Calibrate Threshold (FAR-based)

Use at least 1 hour of background data for reliable false-alarm-rate estimation.

In [ ]:
alarms_per_hour = 1
MIN_CALIBRATION_HOURS = 1.0

cal_spectra = []
cal_total_time = 0.0
for ts in bg_time_series:
    cal_spectra.extend(ts.spectra)
    cal_total_time += sum(s.real_time for s in ts.spectra)
    if cal_total_time >= MIN_CALIBRATION_HOURS * 3600.0:
        break

cal_counts = np.array([s.counts for s in cal_spectra])
cal_real_times = np.array([s.real_time for s in cal_spectra])
cal_live_times = np.array([
    s.live_time if s.live_time is not None else s.real_time
    for s in cal_spectra
])
cal_timestamps = np.cumsum(cal_real_times)
cal_energy_edges = bg_time_series[0].spectra[0].energy_edges

cal_time_series = SpectralTimeSeries.from_array(
    cal_counts,
    energy_edges=cal_energy_edges,
    timestamps=cal_timestamps,
    real_times=cal_real_times,
    live_times=cal_live_times,
)
print(f"Calibration: {len(cal_spectra)} spectra, {cal_total_time/3600:.2f} hrs")

detector.set_threshold_by_far(
    cal_time_series,
    alarms_per_hour=alarms_per_hour,
    verbose=True,
)
print(f"Threshold: {detector.threshold:.6f}")

# Score distribution
all_scores = detector.score_time_series(cal_time_series)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(all_scores, bins=50, alpha=0.7, edgecolor='black')
ax1.axvline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold = {detector.threshold:.2f}')
ax1.set_xlabel('SAD Score')
ax1.set_ylabel('Count')
ax1.set_title('SAD Score Distribution (Background)', fontweight='bold')
ax1.legend()

sorted_scores = np.sort(all_scores)
cdf = np.arange(1, len(sorted_scores)+1) / len(sorted_scores)
ax2.plot(sorted_scores, cdf, lw=2)
ax2.axvline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold = {detector.threshold:.2f}')
ax2.set_xlabel('SAD Score')
ax2.set_ylabel('Cumulative Probability')
ax2.set_title('CDF', fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.show()

## Run Detection on a Source File

In [ ]:
src_files = ds.list_files('source')
test_file = src_files[0]
print(f"Testing on: {test_file}")

test_ts_raw, test_meta = ds.load_file(
    test_file, split='source', detector=DETECTOR, active_only=True,
)

test_ts = test_ts_raw.apply_to_each(lambda s: s.rebin_energy(new_energy_edges))
test_ts = test_ts.rebin_time(INTEGRATION_TIME)

print(f"  {test_ts}")
print(f"  Raw samples: {len(test_ts_raw)}, rebinned: {len(test_ts)}")

sad_scores = detector.process_time_series(test_ts)
summary = detector.get_alarm_summary()

print(f"\nAlarms: {summary['n_alarms']}")
if summary['n_alarms'] > 0:
    print(f"Peak SAD score: {summary['max_peak_metric']:.2f}")
    print(f"Total alarm time: {summary['total_alarm_time']:.2f} s")
for i, alarm in enumerate(detector.alarms, 1):
    print(f"  {i}. {alarm}")

## Visualize Results

In [ ]:
times = test_ts.timestamps
count_rates = np.array([
    float(s.counts.sum()) / float(
        s.live_time if s.live_time is not None else s.real_time
    )
    for s in test_ts.spectra
])

# Source-present regions from the raw (pre-rebin) metadata timestamps
raw_timestamps_ms = test_meta['timestamp'].values.astype(np.float64)
raw_times = (raw_timestamps_ms - raw_timestamps_ms[0]) / 1000.0
source_present = test_meta['is-source-present'].values.astype(bool)
src_changes = np.diff(source_present.astype(int), prepend=0, append=0)
src_starts_idx = np.where(src_changes == 1)[0]
src_ends_idx = np.where(src_changes == -1)[0]
src_start_times = raw_times[src_starts_idx]
src_end_times = raw_times[np.minimum(src_ends_idx, len(raw_times) - 1)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Count rate
ax1.step(times, count_rates, where='post', color='black', lw=0.8,
         label='Count rate')
ax1.set_ylabel(r'Count Rate (s$^{-1}$)', fontsize=12)
ax1.set_title(f'SAD Detection — {test_file}', fontsize=14, fontweight='bold')

for j, (st, et) in enumerate(zip(src_start_times, src_end_times)):
    ax1.axvspan(st, et, alpha=0.15, color='green',
                label='Source present' if j == 0 else '')

for j, alarm in enumerate(detector.alarms):
    ax1.axvspan(alarm.start_time, alarm.end_time, alpha=0.3, color='red',
                label='Alarm' if j == 0 else '')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# SAD scores
ax2.step(times, sad_scores, where='post', color='steelblue', lw=0.8,
         label='SAD score')
ax2.axhline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold ({alarms_per_hour} alarm/hr)')
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('SAD Score', fontsize=12)
ax2.set_title('Spectral Anomaly Metric', fontsize=13, fontweight='bold')
ax2.set_yscale('log')

for j, (st, et) in enumerate(zip(src_start_times, src_end_times)):
    ax2.axvspan(st, et, alpha=0.1, color='green')
for alarm in detector.alarms:
    ax2.axvspan(alarm.start_time, alarm.end_time, alpha=0.2, color='red')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Zoom into the first cluster of source passes
t_zoom = (8500, 10200)
mask = (times >= t_zoom[0]) & (times <= t_zoom[1])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.step(times[mask], count_rates[mask], where='post', color='black', lw=0.8,
         label='Count rate')
ax1.set_ylabel(r'Count Rate (s$^{-1}$)', fontsize=12)
ax1.set_title(f'SAD Detection (zoomed) — {test_file}', fontsize=14, fontweight='bold')

for j, (st, et) in enumerate(zip(src_start_times, src_end_times)):
    if et < t_zoom[0] or st > t_zoom[1]:
        continue
    ax1.axvspan(st, et, alpha=0.15, color='green',
                label='Source present' if j == 0 or st >= t_zoom[0] and j == int(np.searchsorted(src_start_times, t_zoom[0])) else '')

for j, alarm in enumerate(detector.alarms):
    if alarm.end_time < t_zoom[0] or alarm.start_time > t_zoom[1]:
        continue
    ax1.axvspan(alarm.start_time, alarm.end_time, alpha=0.3, color='red',
                label='Alarm' if j == 0 else '')

# Deduplicate legend
handles, labels = ax1.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax1.legend(by_label.values(), by_label.keys(), loc='best')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(t_zoom)

ax2.step(times[mask], sad_scores[mask], where='post', color='steelblue', lw=0.8,
         label='SAD score')
ax2.axhline(detector.threshold, color='red', ls='--', lw=2,
            label=f'Threshold ({alarms_per_hour} alarm/hr)')
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('SAD Score', fontsize=12)
ax2.set_title('Spectral Anomaly Metric (zoomed)', fontsize=13, fontweight='bold')
ax2.set_yscale('log')

for st, et in zip(src_start_times, src_end_times):
    if et < t_zoom[0] or st > t_zoom[1]:
        continue
    ax2.axvspan(st, et, alpha=0.1, color='green')
for alarm in detector.alarms:
    if alarm.end_time < t_zoom[0] or alarm.start_time > t_zoom[1]:
        continue
    ax2.axvspan(alarm.start_time, alarm.end_time, alpha=0.2, color='red')

ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(t_zoom)

plt.tight_layout()
plt.show()